In [3]:
# Need to take a head, relation, tail (h,r,t) triple
# Need to store it on the graph
# Heads and tails are not embeddings. They are simply nodes
# each node has a head_key and a tail_query. the relation is in fact computed at runtime from head_key.tail_query^T
# if a relation is positive, it is navigable.
# if many relations are positive, the largest one is the one that is chosen. If many are large, a random one is chosen.
# When we are trying to combine a relational composite, we instead matmul - composite node key = head_key x tail_query^T
# In this way, we can build an ongoing context. The details of this need to be ironed out. The other options include an ever growing matrix
# of prior relational composites or even just a growing list of head_keys that are dotproducted against each tail_query as you go.
# This has the disadvantage of long chains of thought growing more expensive but obviously it provides the ability to retain early context
# The weights of head_key and tail_query for each node are randomly initialised and the backpropagated with a high learning rate
# when given evidence.
# The question remains about how to make a given relation interpretable.
# If we take a statement "Dogs are mammals" we can decompose into either `Dogs -are> mammals` or `Dogs -> are -> mammals`.
# To me intuitively, the second seems cleaner

In [2]:
import torch
import torch.nn.functional as F
import numpy as np

dim = 8

ModuleNotFoundError: No module named 'torch'

In [104]:
# We use a complex rotation a la RotateE (Sun et al 2019)
# This allows us to meaningfully rotate. It's a cheap operation. It does only give us access to a subset of dim (a toroid within dim) but 
# past work including RotatE has not found the value of more expensive operations as the toroidal subset has been expressive enough
def compose(a, b):
    return a@b

def navigate(a, b):
    gate = torch.sigmoid(torch.dot(a, b))
    return gate * compose(a, b)

def navigate_learn(a, b):
    raw_dot = torch.dot(a, b)
    return raw_dot, navigate(a, b)

In [117]:
dogs_k = torch.randn(dim)
dogs_q = torch.randn(dim)
are_k = torch.randn(dim)
are_q = torch.randn(dim)
mammals_k = torch.randn(dim)
mammals_q = torch.randn(dim)
have_k = torch.randn(dim)
have_q = torch.randn(dim)
teeth_k = torch.randn(dim)
teeth_q = torch.randn(dim)
produce_k = torch.randn(dim)
produce_q = torch.randn(dim)
milk_k = torch.randn(dim)
milk_q = torch.randn(dim)
reptiles_k = torch.randn(dim)
reptiles_q = torch.randn(dim)

# so now, if deciding where to navigate from dogs, we should dotproduct to identify the valid egress pathways

dogs_are = navigate(dogs_k, are_q)
dogs_are_mammals = navigate(dogs_are, mammals_q)
print(dogs_are_mammals.norm()) # should be > 0

mammals_produce = navigate(mammals_k, produce_q)
mammals_produce_milk = navigate(mammals_produce, milk_q)
print(mammals_produce_milk.norm()) # should be > 0

dogs_produce = navigate(dogs_k, produce_q)
dogs_produce_milk = navigate(dogs_produce, milk_q)
print(dogs_produce_milk.norm()) # should be > 0

dogs_are_reptiles = navigate(dogs_are, reptiles_q)
print(dogs_are_reptiles.norm()) # should be < 0

tensor(1.5419e-06)
tensor(20.1837)
tensor(0.0064)
tensor(1.3061e-17)


In [158]:
knowledge = {}
statement_history = {}

def memorise_statement(key: str, value: str, label: bool, key_index: int):
    if key not in statement_history:
        statement_history[key] = {}
    statement_history[key][value] = {
        "label": label,
        "index": key_index
    }
    
def new_node(value: str):
    return {
        "key": torch.randn(dim, requires_grad=True),
        "query": torch.randn(dim, requires_grad=True),
        "value": value,
    }

def learn(statement: str, label: bool):
    print("learning " + statement)
    words = statement.lower().split(" ")
    for i, word in enumerate(words):
        if word not in knowledge:
            knowledge[word] = new_node(word)
        if i > 0:
            memorise_statement(word, statement, label, i)

    training_set = []
    for word in words[1:]:
        training_set.append(statement_history[word])
    
    
    labels = []
    keys_used = set()
    queries_used = set()

    raw_dots = []
    for training_item in training_set:
        for statement, metadata in training_item.items():
            words = statement.lower().split(" ")
            key = knowledge[words[0]]["key"]
            keys_used.add(words[0])
            query = knowledge[words[1]]["query"]
            queries_used.add(words[1])
            raw_dot, composite = navigate_learn(key, query)
            raw_dots.append(raw_dot)
            labels.append(metadata["label"])
            for i in range(2, metadata["index"] + 1):
                queries_used.add(words[i])
                query = knowledge[words[i]]["query"]
                raw_dot, composite = navigate_learn(composite, query)
                raw_dots.append(raw_dot)
                labels.append(metadata["label"])

    # the goal of the training is to take the actual raw dot products (the scores) of each of the transitions and use that as the 
    # source of our cross-entropy used for training our parameters. We need to do not only this, but grab all the out_edges from each of our
    # words and include their raw dot products as well as their logits in the training to avoid catatstrophic forgetting

    params = [knowledge[w]["key"] for w in keys_used] + [knowledge[w]["query"] for w in queries_used]

    
    labels = torch.tensor([float(l) for l in labels])
    scores = torch.stack(raw_dots)
    
    print(scores)
    print(labels)
    optimiser = torch.optim.Adam(params, lr=0.1)
    for _ in range(50):
        optimiser.zero_grad()
        raw_dots = []
        for training_item in training_set:
            for statement, metadata in training_item.items():
                words = statement.lower().split(" ")
                key = knowledge[words[0]]["key"]
                query = knowledge[words[1]]["query"]
                raw_dot, composite = navigate_learn(key, query)
                raw_dots.append(raw_dot)
                for i in range(2, metadata["index"] + 1):
                    query = knowledge[words[i]]["query"]
                    raw_dot, composite = navigate_learn(composite, query)
                    raw_dots.append(raw_dot)
        scores = torch.stack(raw_dots)
        loss = F.binary_cross_entropy_with_logits(scores, labels)
        loss.backward()
        optimiser.step()
        print(loss)
    # here we train with high learning rate. head_tail should be 1. 

learn("Sandwiches are green", False)
learn("Dogs are mammals", True)
learn("Dogs are reptiles", False)
learn("Mammals produce milk", True)
learn("reptiles produce milk", False)
learn("Snakes are reptiles", True)
learn("parrots are multi-coloured", False)

learning Sandwiches are green
tensor([-4.2222, -4.2222, -0.0676], grad_fn=<StackBackward0>)
tensor([0., 0., 0.])
tensor(0.2297, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.3790, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.2310, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.2310, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.2310, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.2310, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.2310, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.2310, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.2310, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.2310, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.2310, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.2310, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.2310, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.2310, grad_fn=<BinaryCrossEntropyWithLogitsBac